# Publishing to TuiML Hub

This tutorial covers how to **publish your own algorithms and datasets** to the TuiML Hub:
- **Algorithms** - Package, configure, and upload ML algorithms
- **Datasets** - Share datasets with the community via CLI or Python
- **Version Management** - Update, version, and manage published content

The Hub is a community-driven ecosystem where developers share ML algorithms and datasets for others to discover, install, and benchmark.

## 1. Hub Ecosystem Overview

TuiML Hub connects algorithm authors with users:

1. **Authors** package algorithms into a standard folder structure
2. **Upload** via the `tuiml upload` CLI commands
3. **Users** discover algorithms with `tuiml.list_algorithms()` and `tuiml.search_algorithms()`
4. **Install** with `remote.install()` or `remote.use()` (covered in `community/01_hub_discover`)
5. **Benchmark** against built-in algorithms with `tuiml.experiment()` (covered in `community/03_hub_benchmark`)

Every published algorithm gets:
- A **detail page** on the Hub web interface
- A **versioned download** endpoint
- **Metadata** (type, category, tags) for search and filtering

## 2. Algorithm Folder Structure

Every Hub algorithm lives in a folder with three required files:

```
my_algorithm/
├── tuiml.json      # Metadata and configuration
├── algorithm.py       # Main algorithm implementation
└── README.md          # Documentation and usage examples
```

You can also include additional files like helper modules, tests, or examples:

```
my_algorithm/
├── tuiml.json
├── algorithm.py
├── README.md
├── utils/
│   ├── __init__.py
│   └── helpers.py
├── tests/
│   └── test_algorithm.py
└── examples/
    └── basic_usage.py
```

## 3. The `tuiml.json` Configuration

The `tuiml.json` file defines your algorithm's metadata. This is what the Hub uses to display, categorize, and manage your algorithm.

In [1]:
import json

# This is the structure of a tuiml.json configuration file
config = {
    "name": "my_random_forest",
    "display_name": "My Random Forest",
    "description": "An optimized random forest with adaptive tree pruning",
    "type": "classifier",
    "category": "ensemble",
    "version": "1.0.0",
    "tags": ["ml", "ensemble", "trees"],
    "main": "algorithm.py",
    "dependencies": ["numpy"]
}

print(json.dumps(config, indent=4))

{
    "name": "my_random_forest",
    "display_name": "My Random Forest",
    "description": "An optimized random forest with adaptive tree pruning",
    "type": "classifier",
    "category": "ensemble",
    "version": "1.0.0",
    "tags": [
        "ml",
        "ensemble",
        "trees"
    ],
    "main": "algorithm.py",
    "dependencies": [
        "numpy"
    ]
}


### Configuration Fields

| Field | Required | Description |
|-------|----------|-------------|
| `name` | Yes | Unique identifier (lowercase, underscores) |
| `display_name` | Yes | Human-readable name shown on Hub |
| `description` | Yes | Brief summary of the algorithm |
| `type` | Yes | One of: `classifier`, `regressor`, `clusterer` |
| `category` | Yes | Grouping: `ensemble`, `trees`, `linear`, `bayesian`, `svm`, `neural`, `neighbors`, `rules`, `clustering` |
| `version` | Yes | Semantic version (e.g., `1.0.0`) |
| `tags` | No | List of searchable tags |
| `main` | Yes | Entry-point Python file |
| `dependencies` | No | Python packages required beyond TuiML |

### Example: Classifier Configuration

In [2]:
# Example: A gradient-boosted classifier
classifier_config = {
    "name": "adaptive_boost_trees",
    "display_name": "Adaptive Boost Trees",
    "description": "Gradient boosting with adaptive learning rates and early stopping",
    "type": "classifier",
    "category": "ensemble",
    "version": "1.0.0",
    "tags": ["boosting", "ensemble", "gradient"],
    "main": "adaptive_boost.py",
    "dependencies": ["numpy", "scipy"]
}

print(json.dumps(classifier_config, indent=4))

{
    "name": "adaptive_boost_trees",
    "display_name": "Adaptive Boost Trees",
    "description": "Gradient boosting with adaptive learning rates and early stopping",
    "type": "classifier",
    "category": "ensemble",
    "version": "1.0.0",
    "tags": [
        "boosting",
        "ensemble",
        "gradient"
    ],
    "main": "adaptive_boost.py",
    "dependencies": [
        "numpy",
        "scipy"
    ]
}


### Example: Regressor Configuration

In [3]:
# Example: A time series regressor
regressor_config = {
    "name": "wavelet_regressor",
    "display_name": "Wavelet Regressor",
    "description": "Regression using wavelet decomposition for feature extraction",
    "type": "regressor",
    "category": "linear",
    "version": "1.0.0",
    "tags": ["regression", "wavelet", "signal"],
    "main": "wavelet_regressor.py",
    "dependencies": ["numpy", "pywt"]
}

print(json.dumps(regressor_config, indent=4))

{
    "name": "wavelet_regressor",
    "display_name": "Wavelet Regressor",
    "description": "Regression using wavelet decomposition for feature extraction",
    "type": "regressor",
    "category": "linear",
    "version": "1.0.0",
    "tags": [
        "regression",
        "wavelet",
        "signal"
    ],
    "main": "wavelet_regressor.py",
    "dependencies": [
        "numpy",
        "pywt"
    ]
}


## 4. Writing `algorithm.py`

Your main algorithm file must define a class that follows the TuiML interface:
- Inherit from `Classifier`, `Regressor`, or `Clusterer`
- Implement `fit(X, y)` and `predict(X)` methods
- Use the `@classifier`, `@regressor`, or `@clusterer` decorator

Here is the standard template:

In [4]:
# This is what algorithm.py should look like
algorithm_template = '''
import numpy as np
from tuiml.base import Classifier, classifier


@classifier(tags=["ensemble", "trees"], version="1.0.0")
class MyRandomForest(Classifier):
    """An optimized random forest with adaptive tree pruning.

    Parameters
    ----------
    n_estimators : int, default=100
        Number of trees in the forest.
    max_depth : int or None, default=None
        Maximum depth of each tree.
    random_state : int or None, default=None
        Random seed for reproducibility.
    """

    def __init__(self, n_estimators=100, max_depth=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state

    def fit(self, X, y):
        """Train the random forest on training data.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Training feature matrix.
        y : np.ndarray of shape (n_samples,)
            Target labels.

        Returns
        -------
        self : MyRandomForest
            Fitted estimator.
        """
        # Your training logic here
        self.classes_ = np.unique(y)
        self.n_features_ = X.shape[1]
        return self

    def predict(self, X):
        """Predict class labels for samples in X.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Feature matrix.

        Returns
        -------
        y_pred : np.ndarray of shape (n_samples,)
            Predicted class labels.
        """
        # Your prediction logic here
        return np.zeros(X.shape[0])
'''

print(algorithm_template)


import numpy as np
from tuiml.base import Classifier, classifier


@classifier(tags=["ensemble", "trees"], version="1.0.0")
class MyRandomForest(Classifier):
    """An optimized random forest with adaptive tree pruning.

    Parameters
    ----------
    n_estimators : int, default=100
        Number of trees in the forest.
    max_depth : int or None, default=None
        Maximum depth of each tree.
    random_state : int or None, default=None
        Random seed for reproducibility.
    """

    def __init__(self, n_estimators=100, max_depth=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state

    def fit(self, X, y):
        """Train the random forest on training data.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Training feature matrix.
        y : np.ndarray of shape (n_samples,)
            Target labels.

        Returns
      

## 5. Writing `README.md`

A good README helps users understand and use your algorithm. Include:

```markdown
# My Random Forest

An optimized random forest with adaptive tree pruning.

## Installation

    tuiml hub install my_random_forest

## Usage

    from tuiml.hub import remote
    model = remote.use("my_random_forest", n_estimators=200)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

## Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| n_estimators | int | 100 | Number of trees |
| max_depth | int | None | Maximum tree depth |
| random_state | int | None | Random seed |

## License

MIT
```

## 6. Creating the Folder with the CLI

The `tuiml upload init` command scaffolds the folder structure and generates a template `tuiml.json` for you:

```bash
# Initialize a new algorithm folder
tuiml upload init ./my_algorithm/

# With a specific name
tuiml upload init ./my_algorithm/ --name my_random_forest
```

This creates:
```
my_algorithm/
├── tuiml.json      # Pre-filled template
├── algorithm.py       # Starter code with base class
└── README.md          # Documentation template
```

Edit the generated files, implement your algorithm, and you are ready to publish.

## 7. Authenticating with the Hub

Before uploading, you need an API token. Get one from the Hub web interface, then authenticate:

```bash
# Option 1: Save token locally (persists across sessions)
tuiml upload login wk_xxxxx

# Option 2: Set environment variable (per-session)
export TUIML_API_TOKEN=wk_xxxxx
```

The token is stored in `~/.tuiml/credentials` and used automatically for all upload commands.

## 8. Uploading Your Algorithm

### 8.1 Dry Run (Preview)

Always start with a dry run to verify what will be uploaded:

```bash
# Preview upload without actually pushing
tuiml upload push ./my_algorithm/ --dry-run
```

This validates the folder structure, checks `tuiml.json`, and shows which files will be included.

### 8.2 Push to Hub

```bash
# Upload the algorithm
tuiml upload push ./my_algorithm/
```

On success, you get a confirmation with the algorithm's Hub URL and ID.

## 9. Updating Published Algorithms

### 9.1 Update Metadata

Change version, description, or tags without re-uploading files:

```bash
# Bump version
tuiml upload update 1 --version "1.1.0"

# Update description
tuiml upload update 1 --description "Improved pruning strategy"

# Add tags
tuiml upload update 1 --tags ml --tags ensemble --tags optimized
```

The algorithm ID (`1` in the examples above) is returned when you first push.

### 9.2 Push a New Version

To upload updated code, change the `version` field in `tuiml.json` and push again:

```bash
# After updating tuiml.json version to "1.1.0"
tuiml upload push ./my_algorithm/
```

The Hub stores each version separately, so users can pin to a specific version.

## 10. Deleting an Algorithm

Remove a published algorithm from the Hub:

```bash
# Delete with confirmation prompt
tuiml upload delete 1

# Skip confirmation
tuiml upload delete 1 --yes
```

**Warning:** Deletion is permanent. Users who installed the algorithm will retain their local copy, but no new installations will be possible.

## 11. Best Practices for Publishing

### Naming
- Use descriptive `name` values: `adaptive_boost_trees` not `abt`
- Keep `display_name` concise but clear: "Adaptive Boost Trees"
- Use lowercase with underscores for `name`

### Versioning
- Follow [Semantic Versioning](https://semver.org/): `MAJOR.MINOR.PATCH`
- Bump `MAJOR` for breaking API changes
- Bump `MINOR` for new features (backward-compatible)
- Bump `PATCH` for bug fixes

### Documentation
- Always include a `README.md` with usage examples
- Document all parameters with types and defaults
- Include at least one working example

### Code Quality
- Follow the TuiML base class interface (`fit`, `predict`)
- Use NumPy-style docstrings (see CLAUDE.md for conventions)
- Include type hints where possible
- List all external dependencies in `tuiml.json`

### Testing
- Add a `tests/` folder with unit tests
- Test on at least one built-in dataset (e.g., iris, diabetes)
- Verify your algorithm works via `remote.use()` after uploading

## 12. CLI Command Reference

| Command | Description |
|---------|-------------|
| `tuiml upload init ./folder/` | Scaffold algorithm folder with templates |
| `tuiml upload login wk_xxxxx` | Save API token for authentication |
| `tuiml upload push ./folder/` | Upload algorithm to Hub |
| `tuiml upload push ./folder/ --dry-run` | Preview upload without pushing |
| `tuiml upload update ID --version "X.Y.Z"` | Update metadata by algorithm ID |
| `tuiml upload delete ID` | Remove algorithm from Hub |
| `tuiml upload delete ID --yes` | Remove without confirmation prompt |

## 13. Publishing Datasets

In addition to algorithms, you can publish datasets to the Hub for others to use in their ML workflows.

### 13.1 Dataset Upload via CLI

Upload datasets directly from the command line:

```bash
# Basic upload
tuiml datasets upload my_data.csv --name "my-dataset" --task classification

# Full upload with all metadata
tuiml datasets upload customer_data.csv \
    --name "customer-churn" \
    --description "Customer churn prediction dataset with usage patterns" \
    --task classification \
    --target "churn" \
    --tags customer --tags marketing --tags telecom
```

Supported formats: CSV, JSON, Parquet, Excel (.xls, .xlsx), ARFF

### 13.2 Dataset Upload via Python

You can also upload datasets programmatically:

In [5]:
from tuiml.hub import datasets
import os

# Set your API token (get from Hub web interface)
# os.environ["TUIML_API_TOKEN"] = "wk_your_token_here"

try:
    # Upload a dataset to the Hub
    info = datasets.upload(
        file_path="customer_data.csv",
        name="customer-churn",
        display_name="Customer Churn Dataset",
        description="Predict customer churn based on usage patterns and demographics",
        task_type="classification",
        target_column="churn",
        tags=["customer", "marketing", "telecom", "churn"],
        readme="""## Customer Churn Dataset

This dataset contains customer usage data for churn prediction.

### Features
- `tenure`: Months with the company
- `monthly_charges`: Monthly bill amount
- `total_charges`: Total amount billed
- `contract`: Contract type (month-to-month, one year, two year)

### Target
- `churn`: Whether the customer churned (Yes/No)
"""
    )
    
    print(f"Dataset uploaded successfully!")
    print(f"  Name: {info.name}")
    print(f"  ID: {info.id}")
    print(f"  Size: {info.rows:,} rows × {info.columns} columns")
    print(f"  URL: http://localhost:8000/dataset/{info.id}")
    
except FileNotFoundError:
    print("Example: File 'customer_data.csv' not found (expected in tutorial)")
except ValueError as e:
    print(f"Example: {e}")
except Exception as e:
    print(f"Example: {e}")

Example: File 'customer_data.csv' not found (expected in tutorial)


### 13.3 Dataset Upload Parameters

| Parameter | Required | Description |
|-----------|----------|-------------|
| `file_path` | Yes | Path to the dataset file |
| `name` | Yes | Unique identifier (lowercase, hyphens) |
| `display_name` | No | Human-readable name (auto-generated if omitted) |
| `description` | No | Brief summary of the dataset |
| `task_type` | No | One of: `classification`, `regression`, `clustering`, `other` |
| `target_column` | No | Name of the target/label column |
| `tags` | No | List of searchable tags |
| `readme` | No | Long description in markdown format |
| `token` | No | API token (uses `TUIML_API_TOKEN` env var if omitted) |

### 13.4 Dataset CLI Command Reference

| Command | Description |
|---------|-------------|
| `tuiml datasets list` | Browse available datasets |
| `tuiml datasets list --task classification` | Filter by task type |
| `tuiml datasets info NAME` | Get detailed dataset info |
| `tuiml datasets download NAME` | Download to local cache |
| `tuiml datasets download NAME -o DIR` | Download to specific directory |
| `tuiml datasets upload FILE --name NAME` | Upload dataset to Hub |